In [1]:
from pathlib import Path
import sys

root = Path.cwd()
while not (root / "src").is_dir() and root != root.parent:
    root = root.parent

if not (root / "src").is_dir():
    raise FileNotFoundError("Could not find the repository folder containing src/")

sys.path.insert(0, str(root))

In [2]:
from qiskit.circuit import QuantumCircuit,ParameterVector
from qiskit_machine_learning.kernels import FidelityQuantumKernel
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,recall_score,f1_score,classification_report
from src.data_load import load_data
from src.preprocess import preprocess
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [3]:
seed=42
qubits=4

X_train, X_test, y_train, y_test= load_data()
X_train_scale,X_test_scale, X_train_pca, X_test_pca, pca=preprocess(X_train,X_test,qubits)

In [4]:
mm = MinMaxScaler(feature_range=(0, np.pi))
X_train_quantum = mm.fit_transform(X_train_pca)
X_test_quantum = np.clip(mm.transform(X_test_pca), 0, np.pi)

In [5]:
x=ParameterVector("x",qubits)
feature_map=QuantumCircuit(qubits)
for i in range(qubits):
    feature_map.ry(x[i],i)

In [6]:
kernel=FidelityQuantumKernel(feature_map=feature_map)
K_train = kernel.evaluate(X_train_quantum, X_train_quantum)
K_test = kernel.evaluate(X_test_quantum, X_train_quantum)
print(K_train.shape,K_test.shape)

(455, 455) (114, 455)


In [7]:
qsvm=SVC(kernel="precomputed",C=1.0)
qsvm.fit(K_train, y_train)
y_pred_q=qsvm.predict(K_test)

In [8]:
print("QSVM accuracy:", accuracy_score(y_test, y_pred_q))
print("Malignant recall:", recall_score(y_test, y_pred_q, pos_label=0))
print("F1 (macro):", f1_score(y_test, y_pred_q, average="macro"))
print(classification_report(y_test, y_pred_q, target_names=["malignant", "benign"]))

QSVM accuracy: 0.956140350877193
Malignant recall: 0.9047619047619048
F1 (macro): 0.9521289997480473
              precision    recall  f1-score   support

   malignant       0.97      0.90      0.94        42
      benign       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [9]:
svm = SVC(kernel="rbf", C=1.0, gamma="scale").fit(X_train_quantum, y_train)
y_pred_classical = svm.predict(X_test_quantum)
print("Classical SVM (same data) accuracy:", accuracy_score(y_test, y_pred_classical))
print("Classical malignant recall:", recall_score(y_test, y_pred_classical, pos_label=0))
print("Classical F1 (macro):", f1_score(y_test, y_pred_classical, average="macro"))

Classical SVM (same data) accuracy: 0.956140350877193
Classical malignant recall: 0.9285714285714286
Classical F1 (macro): 0.9526381387619444


In [10]:
seeds = [0, 1, 2, 3, 4]          # use [0] first to check timing
C_grid = {"C": [0.1, 1, 10, 100, 1000]}
rbf_grid = {"C": [0.1, 1, 10, 100, 1000], "gamma": [0.001, 0.01, 0.1, 1, 10, "scale"]}

def metrics(y_true, pred):
    return (accuracy_score(y_true, pred),
            recall_score(y_true, pred, pos_label=0),
            f1_score(y_true, pred, average="macro"))

In [11]:
rows = []
for seed in seeds:
    X_train, X_test, y_train, y_test = load_data(seed)
    _, _, Xp_tr, Xp_te, _ = preprocess(X_train, X_test, qubits)

    mm = MinMaxScaler(feature_range=(0, np.pi))
    Xtr = mm.fit_transform(Xp_tr)
    Xte = np.clip(mm.transform(Xp_te), 0, np.pi)

    K_tr = kernel.evaluate(Xtr, Xtr)
    K_te = kernel.evaluate(Xte, Xtr)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    q = GridSearchCV(SVC(kernel="precomputed"), C_grid, cv=cv, scoring="f1_macro").fit(K_tr, y_train)
    a, r_, f = metrics(y_test, q.predict(K_te))
    rows.append({"seed": seed, "model": "QSVM angle", "acc": a, "recall_malignant": r_, "f1": f})

    r = GridSearchCV(SVC(kernel="rbf"), rbf_grid, cv=cv, scoring="f1_macro").fit(Xtr, y_train)
    a, r_, f = metrics(y_test, r.predict(Xte))
    rows.append({"seed": seed, "model": "RBF tuned (same input)", "acc": a, "recall_malignant": r_, "f1": f})

    print(f"seed {seed} done", flush=True)
    pd.DataFrame(rows).to_csv("bc_angle.csv", index=False)

df = pd.DataFrame(rows)
print(df.groupby("model")[["acc", "recall_malignant", "f1"]].agg(["mean", "std"]).round(3).to_string())

seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done
                         acc        recall_malignant            f1       
                        mean    std             mean    std   mean    std
model                                                                    
QSVM angle              0.96  0.008            0.929  0.017  0.956  0.008
RBF tuned (same input)  0.96  0.017            0.929  0.024  0.956  0.018
